In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
CSV_DIR = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/csv")
DICOM_ROOT = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/cbis_ddsm")

calc_train = pd.read_csv(CSV_DIR / "calc_case_description_train_set.csv")
calc_test = pd.read_csv(CSV_DIR / "calc_case_description_test_set.csv")
mass_train = pd.read_csv(CSV_DIR / "mass_case_description_train_set.csv")
mass_test = pd.read_csv(CSV_DIR / "mass_case_description_test_set.csv")

In [5]:
# Padronização e Concatenação
for df_name, df in [('calc_train', calc_train), ('calc_test', calc_test), 
                    ('mass_train', mass_train), ('mass_test', mass_test)]:
    df['set'] = 'Training' if 'train' in df_name else 'Test'
    df['lesion_type'] = 'Calcification' if 'calc' in df_name else 'Mass'

df = pd.concat([calc_train, calc_test, mass_train, mass_test], ignore_index=True)

# Simplificação de BENIGN_WITHOUT_CALLBACK para BENIGN
df['pathology_simple'] = df['pathology'].replace({'BENIGN_WITHOUT_CALLBACK': 'BENIGN'})

In [13]:
print("1. VISÃO GERAL DO DATASET")
print(f"Casos (tumores) totais: {len(df)}")
print(f"Pacientes únicos: {df['patient_id'].nunique()}\n")

tabela_cruzada = pd.crosstab(index=df['lesion_type'], columns=[df['set'], df['pathology_simple']], margins=True)
display(tabela_cruzada)

print("\n2. TUMORES POR PACIENTE")
tumores_por_paciente = df.groupby('patient_id')['abnormality id'].nunique()
print(f"Média de tumores por paciente: {tumores_por_paciente.mean():.2f}")
print(f"Pacientes com apenas 1 tumor: {(tumores_por_paciente == 1).sum()}")
print(f"Pacientes com 2+ tumores:     {(tumores_por_paciente >= 2).sum()}")

print("\n3. VERIFICAÇÃO DE VISTAS")
vistas_por_mama = df.groupby(['patient_id', 'left or right breast'])['image view'].apply(set)
tem_ambas = vistas_por_mama.apply(lambda x: 'CC' in x and 'MLO' in x)
print(f"Mamografias com AMBAS as vistas: {tem_ambas.sum()}")
print(f"Mamografias SEM ambas as vistas: {(~tem_ambas).sum()}")

print("\n4. VERIFICAÇÃO DE VAZAMENTO (TRAIN vs TEST)")

# Encontra pacientes que estão em ambos os conjuntos
pacientes_train = set(df[df['set'] == 'Training']['patient_id'])
pacientes_test = set(df[df['set'] == 'Test']['patient_id'])
pacientes_ambos = pacientes_train & pacientes_test

print(f"Pacientes exclusivos no Training: {len(pacientes_train - pacientes_test)}")
print(f"Pacientes exclusivos no Test:     {len(pacientes_test - pacientes_train)}")
print(f"Pacientes em AMBOS:               {len(pacientes_ambos)}")

# Verifica se o MESMO tumor (abnormality id) está em ambos
df_ambos = df[df['patient_id'].isin(pacientes_ambos)]
vazamento_real = []

for paciente in pacientes_ambos:
    sub = df_ambos[df_ambos['patient_id'] == paciente]
    ids_train = set(sub[sub['set'] == 'Training']['abnormality id'])
    ids_test = set(sub[sub['set'] == 'Test']['abnormality id'])
    
    repetidos = ids_train & ids_test
    if repetidos:
        vazamento_real.append({'Paciente': paciente, 'Tumores Repetidos': list(repetidos)})

if vazamento_real:
    print(f"\nATENÇÃO: {len(vazamento_real)} pacientes possuem o MESMO tumor no treino e no teste.")
    display(pd.DataFrame(vazamento_real))
else:
    print("\nNenhum tumor repetido entre Train e Test.")

print("\n5. DISPONIBILIDADE DE IMAGENS")
pacientes_completas, pacientes_roi = set(), set()

for folder in DICOM_ROOT.iterdir():
    if not folder.is_dir() or ('Mass-' not in folder.name and 'Calc-' not in folder.name):
        continue
        
    partes = folder.name.split('_P_')
    if len(partes) < 2: continue
    
    paciente_id = 'P_' + partes[1].split('_')[0]
    
    # No CBIS-DDSM, pastas que terminam em dígito são ROIs (recortes)
    if folder.name[-1].isdigit():
        pacientes_roi.add(paciente_id)
    else:
        pacientes_completas.add(paciente_id)

print(f"Pacientes com mamografia completa: {len(pacientes_completas)}")
print(f"Pacientes com ROI:                 {len(pacientes_roi)}")
print(f"Pacientes APENAS com ROI (sem completa): {len(pacientes_roi - pacientes_completas)}")

1. VISÃO GERAL DO DATASET
Casos (tumores) totais: 3568
Pacientes únicos: 1566



set                Test           Training             All
pathology_simple BENIGN MALIGNANT   BENIGN MALIGNANT      
lesion_type                                               
Calcification       197       129     1002       544  1872
Mass                231       147      681       637  1696
All                 428       276     1683      1181  3568


2. TUMORES POR PACIENTE
Média de tumores por paciente: 1.18
Pacientes com apenas 1 tumor: 1395
Pacientes com 2+ tumores:     171

3. VERIFICAÇÃO DE VISTAS
Mamografias com AMBAS as vistas: 1324
Mamografias SEM ambas as vistas: 384

4. VERIFICAÇÃO DE VAZAMENTO (TRAIN vs TEST)
Pacientes exclusivos no Training: 1217
Pacientes exclusivos no Test:     318
Pacientes em AMBOS:               31

ATENÇÃO: 19 pacientes possuem o MESMO tumor no treino e no teste.


,Paciente,Tumores Repetidos
0,P_00969,[3]
1,P_00652,[1]
2,P_00394,[1]
3,P_01621,[1]
4,P_01605,[2]
5,P_00510,[2]
6,P_00924,[1]
7,P_00495,"[1, 2]"
8,P_00639,[1]
9,P_01237,[1]



5. DISPONIBILIDADE DE IMAGENS
Pacientes com mamografia completa: 1460
Pacientes com ROI:                 1530
Pacientes APENAS com ROI (sem completa): 78
